In [7]:
import numpy as np
import pandas as pd
import scipy.sparse as sps
import dimod
import uuid
import os

from sklearn.model_selection import train_test_split
from dwave.samplers import SimulatedAnnealingSampler
from qclef import qa_access as qa

# Framework imports
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender


# =========================================================
# LOAD DATA
# =========================================================

n_features = 400

URM_all_df = pd.read_csv(
    "/config/workspace/datasets/Tasks/1B/feature_selection_dataset_URM_train.csv"
)

ICM_df = pd.read_csv(
    "/config/workspace/datasets/Tasks/1B/feature_selection_dataset_400_ICM.csv"
)


# =========================================================
# USERS / ITEMS
# =========================================================

n_users = URM_all_df["UserID"].max() + 1
n_items = ICM_df["ItemID"].max() + 1


# =========================================================
# DATAFRAME -> SPARSE
# =========================================================

def _from_df_to_sparse(URM_df, n_users, n_items):

    URM_sps = sps.csr_matrix(
        (
            np.ones(len(URM_df)),
            (
                URM_df["UserID"].values,
                URM_df["ItemID"].values
            )
        ),
        shape=(n_users, n_items)
    )

    return URM_sps


# =========================================================
# BUILD MATRICES
# =========================================================

URM_all = _from_df_to_sparse(
    URM_all_df,
    n_users,
    n_items
)

ICM = sps.csr_matrix(
    (
        ICM_df["Value"].values,
        (
            ICM_df["ItemID"].values,
            ICM_df["FeatureID"].values
        )
    ),
    shape=(n_items, n_features)
)


# =========================================================
# TRAIN / VALIDATION SPLIT
# =========================================================

URM_train_df, URM_validation_df = train_test_split(
    URM_all_df,
    test_size=0.20,
    random_state=42
)

URM_train = _from_df_to_sparse(
    URM_train_df,
    n_users,
    n_items
)

URM_validation = _from_df_to_sparse(
    URM_validation_df,
    n_users,
    n_items
)


# =========================================================
# EVALUATOR
# =========================================================

evaluator_validation = EvaluatorHoldout(
    URM_validation,
    cutoff_list=[10]
)


# =========================================================
# SIMILARITIES
# =========================================================

similarity_collaborative = URM_train.T.dot(URM_train)

similarity_content = ICM.dot(ICM.T)

similarity_collaborative_bin = similarity_collaborative.astype(bool)

similarity_content_bin = similarity_content.astype(bool)

Keep = similarity_collaborative_bin.multiply(
    similarity_content_bin
)


# =========================================================
# FEATURE PENALIZATION MATRIX
# =========================================================

FPM = ICM.T.dot(Keep).dot(ICM)


# =========================================================
# CREATE BQM
# =========================================================

BQM = dimod.BinaryQuadraticModel(
    FPM.toarray(),
    "BINARY"
)

BQM.normalize()


# =========================================================
# CONSTRAINT
# =========================================================

k_largest = 200
penalty = 0.01

BQM_k = (
    dimod.generators.combinations(
        BQM.num_variables,
        k_largest
    ) * penalty
)

BQM_k.update(BQM)


# =========================================================
# QCLEF TRACKED SAMPLER
# =========================================================

sampler = SimulatedAnnealingSampler()

sampleset = qa.submit(
    sampler,
    SimulatedAnnealingSampler.sample,
    BQM_k,
    label="1B Feature Selection",
    num_reads=1000
)


# =========================================================
# TRACKING INFO
# =========================================================

print(sampleset)
print(sampleset.info)

best_solution = sampleset.first.sample

problem_id = sampleset.info.get(
    "problem_id",
    str(uuid.uuid4())
)

print("Problem ID:", problem_id)


# =========================================================
# FILTER ICM
# =========================================================

def _filter_ICM(ICM_all, selection_dict):

    selected_flag = np.zeros(
        ICM_all.shape[1],
        dtype=bool
    )

    for key, value in selection_dict.items():

        if value == 1:
            selected_flag[int(key)] = True

    selected_ICM = (
        ICM_all.tocsc()[:, selected_flag]
        .tocsr()
    )

    return selected_ICM, selected_flag.sum()


ICM_selected, n_selected = _filter_ICM(
    ICM,
    best_solution
)

print("Selected Features:", n_selected)


# =========================================================
# TRAIN RECOMMENDER
# =========================================================

recommender_instance = ItemKNNCBFRecommender(
    URM_train,
    ICM_selected
)

recommender_instance.fit(
    topK=100,
    shrink=5,
    similarity='cosine',
    normalize=True
)


# =========================================================
# EVALUATION
# =========================================================

result_df, result_string = (
    evaluator_validation.evaluateRecommender(
        recommender_instance
    )
)

print(result_string)
  
print(
    "The NDCG@10 is {:.4f}".format(
        result_df.loc[10, "NDCG"]
    )
)


# =========================================================
# CREATE SUBMISSION FILE
# =========================================================

selected_features = [
    int(k)
    for k, v in best_solution.items()
    if v == 1
]

selected_features = sorted(selected_features)

problem_ids = [problem_id]




EvaluatorHoldout: Ignoring 3 ( 0.0%) Users that have less than 1 test interactions
     0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 ... 399   energy num_oc.
478  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   0 0.522625       1
926  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   1 0.524304       1
44   0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   1 0.525065       1
909  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   0 0.530522       1
549  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   1  0.53151       1
974  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   0 0.533487       1
266  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   1 0.533758       1
693  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   1 0.533882       1
856  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   1 0.534758       1
651  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0 ...   0 0.534945       1
923  0  0  0  0  0  0  0  0  0  0  0 

In [8]:
GROUP_NAME = "NUCES-FAST"
SUBMISSION_ID = "Ensemble"

METHOD = "SA"
TASK = "1B"
DATASET = "500_ICM"


filename = (
    f"{TASK}_{DATASET}_{METHOD}_"
    f"{GROUP_NAME}_{SUBMISSION_ID}.txt"
)

output_dir = "/config/workspace/submissions"

os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(
    output_dir,
    filename
)


with open(file_path, "w") as f:

    # selected features
    for feat in selected_features:
        f.write(f"{feat}\n")

    # last line = tracked problem IDs
    f.write(f"{problem_ids}\n")


print("Submission saved to:", file_path)
print("Tracked Problem IDs:", problem_ids)

Submission saved to: /config/workspace/submissions/1B_500_ICM_QA_NUCES-FAST_Ensemble.txt
Tracked Problem IDs: ['SA-7049']


In [9]:
import numpy as np
import pandas as pd
import scipy.sparse as sps
import dimod
import uuid
import os

from sklearn.model_selection import train_test_split
from dwave.samplers import SimulatedAnnealingSampler
from qclef import qa_access as qa

from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender


# =========================================================
# LOAD DATA
# =========================================================

n_features = 400

URM_all_df = pd.read_csv(
    "/config/workspace/datasets/Tasks/1B/feature_selection_dataset_URM_train.csv"
)

ICM_df = pd.read_csv(
    "/config/workspace/datasets/Tasks/1B/feature_selection_dataset_400_ICM.csv"
)

n_users = URM_all_df["UserID"].max() + 1
n_items = ICM_df["ItemID"].max() + 1


def to_sparse(df, n_users, n_items):
    return sps.csr_matrix(
        (
            np.ones(len(df)),
            (df["UserID"].values, df["ItemID"].values)
        ),
        shape=(n_users, n_items)
    )


URM_all = to_sparse(URM_all_df, n_users, n_items)

ICM = sps.csr_matrix(
    (
        ICM_df["Value"].values,
        (ICM_df["ItemID"].values, ICM_df["FeatureID"].values)
    ),
    shape=(n_items, n_features)
)


URM_train_df, URM_validation_df = train_test_split(
    URM_all_df,
    test_size=0.2,
    random_state=42
)

URM_train = to_sparse(URM_train_df, n_users, n_items)
URM_validation = to_sparse(URM_validation_df, n_users, n_items)


evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[10])


# =========================================================
# STEP 1: CF SIGNAL (USER-AWARE)
# =========================================================

item_pop = np.asarray(URM_train.sum(axis=0)).ravel()

feature_cf = ICM.T.dot(item_pop)
feature_cf = feature_cf / (feature_cf.max() + 1e-9)


# =========================================================
# STEP 2: FEATURE DISCRIMINATION (IMPORTANT)
# =========================================================

good_items = item_pop > np.percentile(item_pop, 70)
bad_items = item_pop < np.percentile(item_pop, 30)

good_signal = np.asarray(ICM[good_items].sum(axis=0)).ravel()
bad_signal = np.asarray(ICM[bad_items].sum(axis=0)).ravel()

feature_sep = good_signal - bad_signal
feature_sep = feature_sep / (np.abs(feature_sep).max() + 1e-9)


# =========================================================
# STEP 3: COVERAGE SIGNAL (FIXES YOUR MAIN ISSUE)
# =========================================================

item_degree = np.asarray((ICM > 0).sum(axis=1)).ravel()

item_degree = item_degree / (item_degree.max() + 1e-9)

feature_coverage = ICM.T.dot(item_degree)
feature_coverage = np.asarray(feature_coverage).ravel()
feature_coverage = feature_coverage / (feature_coverage.max() + 1e-9)


# =========================================================
# STEP 4: REDUNDANCY (LIGHTWEIGHT ONLY)
# =========================================================

F = ICM.T.dot(ICM)
F.setdiag(0)

redundancy = np.asarray(F.sum(axis=1)).ravel()
redundancy = redundancy / (redundancy.max() + 1e-9)


# =========================================================
# STEP 5: FINAL QUBO
# =========================================================

Q = {}

alpha = 1.8   # CF importance (VERY IMPORTANT)
beta = 1.3    # discrimination
gamma = 1.0   # redundancy penalty
delta = 1.1   # coverage boost

for i in range(n_features):

    score = (
        alpha * feature_cf[i]
        + beta * feature_sep[i]
        + delta * feature_coverage[i]
        - gamma * redundancy[i]
    )

    Q[(i, i)] = -score


# =========================================================
# STEP 6: BQM + STRONG CONSTRAINT (FIXES COLLAPSE)
# =========================================================

BQM = dimod.BinaryQuadraticModel.from_qubo(Q)

# IMPORTANT FIX: do NOT select too few features
k = 200   # critical fix (you were collapsing at 32)

BQM.update(
    dimod.generators.combinations(n_features, k)
)


# =========================================================
# STEP 7: SOLVE (SA + QCLEF)
# =========================================================

sampler = SimulatedAnnealingSampler()

sampleset = qa.submit(
    sampler,
    SimulatedAnnealingSampler.sample,
    BQM,
    label="1B_Ensemble_500",
    num_reads=2000
)

best = sampleset.first.sample

problem_id = sampleset.info.get(
    "problem_id",
    str(uuid.uuid4())
)


# =========================================================
# STEP 8: SELECT FEATURES
# =========================================================

selected_features = sorted([
    i for i, v in best.items() if v == 1
])

print("Selected features:", len(selected_features))


# =========================================================
# STEP 9: MODEL
# =========================================================

ICM_selected = ICM.tocsc()[:, selected_features].tocsr()

model = ItemKNNCBFRecommender(
    URM_train,
    ICM_selected
)

model.fit(
    topK=100,
    shrink=5,
    similarity='cosine',
    normalize=True
)


# =========================================================
# STEP 10: EVALUATION
# =========================================================

result_df, result_string = evaluator.evaluateRecommender(model)

print(result_string)

print("NDCG@10 =", result_df.loc[10, "NDCG"])




EvaluatorHoldout: Ignoring 3 ( 0.0%) Users that have less than 1 test interactions
Selected features: 200
ItemKNNCBFRecommender: URM Detected 23 ( 0.2%) items with no interactions.
ItemKNNCBFRecommender: ICM Detected 98 ( 0.7%) items with no features.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 14607 (100.0%), 3203.75 column/sec. Elapsed time 4.56 sec
EvaluatorHoldout: Processed 20425 (100.0%) in 9.76 sec. Users per second: 2092
CUTOFF: 10 - PRECISION: 0.0378458, PRECISION_RECALL_MIN_DEN: 0.0391570, RECALL: 0.0140324, MAP: 0.0156724, MAP_MIN_DEN: 0.0161519, MRR: 0.1062751, NDCG: 0.0416678, F1: 0.0204736, HIT_RATE: 0.2641860, ARHR_ALL_HITS: 0.1280165, NOVELTY: 0.0094624, AVERAGE_POPULARITY: 0.0842341, DIVERSITY_MEAN_INTER_LIST: 0.9796469, DIVERSITY_HERFINDAHL: 0.9979599, COVERAGE_ITEM: 0.5733552, COVERAGE_ITEM_HIT: 0.0710618, ITEMS_IN_GT: 0.9141507, COVERAGE_USER: 0.9998531, COVERAGE_USER_HIT: 0.2641472, USERS_IN_GT: 0.9998531, DIVERSITY_GINI: 0.11171

In [10]:
# =========================================================
# STEP 10: SUBMISSION FILE
# =========================================================

GROUP_NAME = "NUCES-FAST"
SUBMISSION_ID = "Ensemble"
METHOD = "SA"
TASK = "1B"
DATASET = "400_ICM"

filename = f"{TASK}_{DATASET}_{METHOD}_{GROUP_NAME}_{SUBMISSION_ID}.txt"

output_dir = "/config/workspace/submissions"

import os
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, filename)

with open(file_path, "w") as f:

    for feat in selected_features:
        f.write(f"{feat}\n")

    f.write(f"{[problem_id]}\n")

print("Saved:", file_path)
print("Problem ID:", problem_id)

Saved: /config/workspace/submissions/1B_400_ICM_SA_NUCES-FAST_Ensemble.txt
Problem ID: SA-7050


# Quantum Annealing

In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sps
import dimod
import uuid
import os

from sklearn.model_selection import train_test_split

from dwave.samplers import SimulatedAnnealingSampler
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.preprocessing import ScaleComposite

from qclef import qa_access as qa

from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender


# =========================================================
# QUANTUM ANNEALING FUNCTION
# =========================================================

def run_quantum_annealing(bqm):

    print("Running Quantum Annealing...")

    task_label = "1B_QA_400_SPARSE"

    try:

        sampler = EmbeddingComposite(
            ScaleComposite(
                DWaveSampler()
            )
        )

        sampleset = qa.submit(
            sampler,
            EmbeddingComposite.sample,
            bqm,
            label=task_label,
            num_reads=100,
            chain_strength=2.0
        )

        problem_id = sampleset.info.get(
            "problem_id",
            str(uuid.uuid4())
        )

        print("Quantum Annealing completed.")
        print("Problem ID:", problem_id)

        return sampleset, [problem_id]

    except Exception as e:

        print("Quantum run failed.")
        print("Reason:", e)

        print("Falling back to Simulated Annealing...")

        sampler = SimulatedAnnealingSampler()

        sampleset = qa.submit(
            sampler,
            SimulatedAnnealingSampler.sample,
            bqm,
            label="1B_SA_FALLBACK",
            num_reads=2000
        )

        problem_id = sampleset.info.get(
            "problem_id",
            str(uuid.uuid4())
        )

        print("SA completed.")
        print("Problem ID:", problem_id)

        return sampleset, [problem_id]


# =========================================================
# LOAD DATA
# =========================================================

n_features_original = 400

URM_all_df = pd.read_csv(
    "/config/workspace/datasets/Tasks/1B/feature_selection_dataset_URM_train.csv"
)

ICM_df = pd.read_csv(
    "/config/workspace/datasets/Tasks/1B/feature_selection_dataset_400_ICM.csv"
)

n_users = URM_all_df["UserID"].max() + 1
n_items = ICM_df["ItemID"].max() + 1


# =========================================================
# SPARSE CONVERSION
# =========================================================

def to_sparse(df, n_users, n_items):

    return sps.csr_matrix(
        (
            np.ones(len(df)),
            (df["UserID"].values, df["ItemID"].values)
        ),
        shape=(n_users, n_items)
    )


URM_all = to_sparse(
    URM_all_df,
    n_users,
    n_items
)

ICM = sps.csr_matrix(
    (
        ICM_df["Value"].values,
        (
            ICM_df["ItemID"].values,
            ICM_df["FeatureID"].values
        )
    ),
    shape=(n_items, n_features_original)
)

ICM = ICM.tocsr()


# =========================================================
# TRAIN / VALIDATION SPLIT
# =========================================================

URM_train_df, URM_validation_df = train_test_split(
    URM_all_df,
    test_size=0.2,
    random_state=42
)

URM_train = to_sparse(
    URM_train_df,
    n_users,
    n_items
)

URM_validation = to_sparse(
    URM_validation_df,
    n_users,
    n_items
)

evaluator = EvaluatorHoldout(
    URM_validation,
    cutoff_list=[10]
)


# =========================================================
# STEP 1: PRESELECTION (IMPORTANT)
# REDUCE 400 -> 150 FEATURES
# =========================================================

item_pop = np.asarray(
    URM_train.sum(axis=0)
).ravel()

feature_cf_full = ICM.T.dot(item_pop)
feature_cf_full = np.asarray(feature_cf_full).ravel()

top_features = np.argsort(feature_cf_full)[-150:]

ICM_reduced = ICM[:, top_features]

n_features = ICM_reduced.shape[1]

print("Reduced features:", n_features)


# =========================================================
# STEP 2: SIGNALS
# =========================================================

feature_cf = feature_cf_full[top_features]

feature_cf = feature_cf / (
    feature_cf.max() + 1e-9
)

good_items = item_pop > np.percentile(item_pop, 70)
bad_items = item_pop < np.percentile(item_pop, 30)

good_signal = np.asarray(
    ICM_reduced[good_items, :].sum(axis=0)
).ravel()

bad_signal = np.asarray(
    ICM_reduced[bad_items, :].sum(axis=0)
).ravel()

feature_sep = good_signal - bad_signal

feature_sep = feature_sep / (
    np.abs(feature_sep).max() + 1e-9
)

item_degree = np.asarray(
    (ICM_reduced > 0).sum(axis=1)
).ravel()

item_degree = item_degree / (
    item_degree.max() + 1e-9
)

feature_coverage = ICM_reduced.T.dot(item_degree)

feature_coverage = np.asarray(
    feature_coverage
).ravel()

feature_coverage = feature_coverage / (
    feature_coverage.max() + 1e-9
)


# =========================================================
# STEP 3: SPARSE REDUNDANCY
# =========================================================

F = ICM_reduced.T.dot(ICM_reduced).tocoo()

redundancy = np.zeros(n_features)

for i, j, v in zip(F.row, F.col, F.data):

    if i != j:
        redundancy[i] += v

redundancy = redundancy / (
    redundancy.max() + 1e-9
)


# =========================================================
# STEP 4: BUILD SPARSE QUBO
# =========================================================

Q = {}

alpha = 1.8
beta = 1.3
gamma = 0.7
delta = 1.1

for i in range(n_features):

    score = (
        alpha * feature_cf[i]
        + beta * feature_sep[i]
        + delta * feature_coverage[i]
        - gamma * redundancy[i]
    )

    Q[(i, i)] = -score


# =========================================================
# STEP 5: ADD ONLY SPARSE INTERACTIONS
# VERY IMPORTANT FOR QPU EMBEDDING
# =========================================================

F = F.tocoo()

threshold = np.percentile(F.data, 99)

for i, j, v in zip(F.row, F.col, F.data):

    if i >= j:
        continue

    if v > threshold:

        interaction_strength = 0.03 * (
            v / (F.data.max() + 1e-9)
        )

        Q[(i, j)] = interaction_strength


# =========================================================
# STEP 6: BUILD BQM
# =========================================================

BQM = dimod.BinaryQuadraticModel.from_qubo(Q)


# =========================================================
# STEP 7: SOFT CARDINALITY CONSTRAINT
# MUCH MORE QPU FRIENDLY
# =========================================================
k = 75

lambda_penalty = 0.015

for i in range(n_features):

    BQM.add_variable(
        i,
        lambda_penalty * (1 - 2 * k)
    )

for i in range(n_features):

    for j in range(i + 1, n_features):

        if (i, j) in Q:

            BQM.add_interaction(
                i,
                j,
                2 * lambda_penalty
            )


# =========================================================
# STEP 8: QUANTUM ANNEALING
# =========================================================

sampleset, problem_ids = run_quantum_annealing(BQM)

best = sampleset.first.sample


# =========================================================
# STEP 9: RECOVER ORIGINAL FEATURE IDS
# =========================================================

selected_reduced = [
    i for i, v in best.items()
    if v == 1
]

selected_features = sorted([
    int(top_features[i])
    for i in selected_reduced
])

print("Selected features:", len(selected_features))


# =========================================================
# STEP 10: MODEL TRAINING
# =========================================================

ICM_selected = ICM.tocsc()[
    :,
    selected_features
].tocsr()

model = ItemKNNCBFRecommender(
    URM_train,
    ICM_selected
)

model.fit(
    topK=100,
    shrink=5,
    similarity='cosine',
    normalize=True
)


# =========================================================
# STEP 11: EVALUATION
# =========================================================

result_df, result_string = evaluator.evaluateRecommender(
    model
)

print(result_string)

print(
    "NDCG@10 =",
    result_df.loc[10, "NDCG"]
)


# =========================================================
# STEP 12: SAVE SUBMISSION
# =========================================================

GROUP_NAME = "NUCES_FAST"

SUBMISSION_ID = "QA_400"

METHOD = "QA"

TASK = "1B"

DATASET = "400_ICM"

filename = (
    f"{TASK}_{DATASET}_{METHOD}_"
    f"{GROUP_NAME}_{SUBMISSION_ID}.txt"
)

output_dir = "/config/workspace/submissions"

os.makedirs(
    output_dir,
    exist_ok=True
)

file_path = os.path.join(
    output_dir,
    filename
)

with open(file_path, "w") as f:

    for feat in selected_features:
        f.write(f"{feat}\n")

    f.write(f"{problem_ids}\n")

print("\nSubmission saved to:")
print(file_path)

EvaluatorHoldout: Ignoring 3 ( 0.0%) Users that have less than 1 test interactions
Reduced features: 150
Running Quantum Annealing...
Quantum Annealing completed.
Problem ID: c77834cd-4757-4927-bd84-fc7510e037e8
Selected features: 150
ItemKNNCBFRecommender: URM Detected 23 ( 0.2%) items with no interactions.
ItemKNNCBFRecommender: ICM Detected 98 ( 0.7%) items with no features.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 14607 (100.0%), 3167.38 column/sec. Elapsed time 4.61 sec
EvaluatorHoldout: Processed 20425 (100.0%) in 9.38 sec. Users per second: 2178
CUTOFF: 10 - PRECISION: 0.0414884, PRECISION_RECALL_MIN_DEN: 0.0430559, RECALL: 0.0152094, MAP: 0.0168190, MAP_MIN_DEN: 0.0173877, MRR: 0.1088815, NDCG: 0.0446151, F1: 0.0222588, HIT_RATE: 0.2794125, ARHR_ALL_HITS: 0.1338492, NOVELTY: 0.0093095, AVERAGE_POPULARITY: 0.0893689, DIVERSITY_MEAN_INTER_LIST: 0.9714605, DIVERSITY_HERFINDAHL: 0.9971413, COVERAGE_ITEM: 0.5702061, COVERAGE_ITEM_HIT: 0.0655850

# Quantum Annealer Summary 

EvaluatorHoldout: Ignoring 3 ( 0.0%) Users that have less than 1 test interactions
Running Quantum Annealing...
Quantum Annealing completed.
Problem ID: c2a4c334-8a32-4261-80d3-91bd6ee83fd7
Selected features: 72
ItemKNNCBFRecommender: URM Detected 23 ( 0.2%) items with no interactions.
ItemKNNCBFRecommender: ICM Detected 100 ( 0.7%) items with no features.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 14607 (100.0%), 3227.48 column/sec. Elapsed time 4.53 sec
EvaluatorHoldout: Processed 20425 (100.0%) in 10.33 sec. Users per second: 1978
CUTOFF: 10 - PRECISION: 0.0298409, PRECISION_RECALL_MIN_DEN: 0.0308937, RECALL: 0.0107471, MAP: 0.0116478, MAP_MIN_DEN: 0.0119854, MRR: 0.0830103, NDCG: 0.0322474, F1: 0.0158029, HIT_RATE: 0.2211995, ARHR_ALL_HITS: 0.0977163, NOVELTY: 0.0094072, AVERAGE_POPULARITY: 0.0851227, DIVERSITY_MEAN_INTER_LIST: 0.9752308, DIVERSITY_HERFINDAHL: 0.9975183, COVERAGE_ITEM: 0.5617170, COVERAGE_ITEM_HIT: 0.0637366, ITEMS_IN_GT: 0.9141507, COVERAGE_USER: 0.9998531, COVERAGE_USER_HIT: 0.2211670, USERS_IN_GT: 0.9998531, DIVERSITY_GINI: 0.1052048, SHANNON_ENTROPY: 10.4923544, RATIO_DIVERSITY_HERFINDAHL: 0.9981389, RATIO_DIVERSITY_GINI: 0.5533270, RATIO_SHANNON_ENTROPY: 0.8972351, RATIO_AVERAGE_POPULARITY: 0.3491612, RATIO_NOVELTY: 0.0934806, 

NDCG@10 = 0.03224735073157422

Submission saved to:
/config/workspace/submissions/1B_100_ICM_QA_NUCES_FAST_Ensemble.txt

In [28]:
import numpy as np
import pandas as pd
import scipy.sparse as sps
import dimod
import uuid

from sklearn.model_selection import train_test_split
from dwave.samplers import SimulatedAnnealingSampler
from qclef import qa_access as qa

from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender


# =========================================================
# LOAD DATA
# =========================================================

n_features = 100

URM_all_df = pd.read_csv(
    "/config/workspace/datasets/Tasks/1B/feature_selection_dataset_URM_train.csv"
)

ICM_df = pd.read_csv(
    "/config/workspace/datasets/Tasks/1B/feature_selection_dataset_100_ICM.csv"
)

n_users = URM_all_df["UserID"].max() + 1
n_items = ICM_df["ItemID"].max() + 1


def to_sparse(df, n_users, n_items):
    return sps.csr_matrix(
        (np.ones(len(df)), (df["UserID"], df["ItemID"])),
        shape=(n_users, n_items)
    )


URM_all = to_sparse(URM_all_df, n_users, n_items)

ICM = sps.csr_matrix(
    (
        ICM_df["Value"].values,
        (ICM_df["ItemID"].values, ICM_df["FeatureID"].values)
    ),
    shape=(n_items, n_features)
)


URM_train_df, URM_validation_df = train_test_split(
    URM_all_df,
    test_size=0.2,
    random_state=42
)

URM_train = to_sparse(URM_train_df, n_users, n_items)
URM_validation = to_sparse(URM_validation_df, n_users, n_items)

evaluator = EvaluatorHoldout(URM_validation, cutoff_list=[10])


# =========================================================
# PRECOMPUTE SIGNALS (FIXED ONCE)
# =========================================================

item_pop = np.asarray(URM_train.sum(axis=0)).ravel()

feature_cf = ICM.T.dot(item_pop)
feature_cf = feature_cf / (feature_cf.max() + 1e-9)

good_items = item_pop > np.percentile(item_pop, 70)
bad_items = item_pop < np.percentile(item_pop, 30)

feature_sep = (
    np.asarray(ICM[good_items].sum(axis=0)).ravel()
    - np.asarray(ICM[bad_items].sum(axis=0)).ravel()
)
feature_sep = feature_sep / (np.abs(feature_sep).max() + 1e-9)

item_degree = np.asarray((ICM > 0).sum(axis=1)).ravel()
item_degree = item_degree / (item_degree.max() + 1e-9)

feature_cov = ICM.T.dot(item_degree)
feature_cov = np.asarray(feature_cov).ravel()
feature_cov = feature_cov / (feature_cov.max() + 1e-9)

F = ICM.T.dot(ICM)
F.setdiag(0)

redundancy = np.asarray(F.sum(axis=1)).ravel()
redundancy = redundancy / (redundancy.max() + 1e-9)


# =========================================================
# TUNING GRID (IMPORTANT PART)
# =========================================================

param_grid = [
    (2.5, 1.2, 1.0, 1.0),
    (2.0, 1.0, 1.2, 1.0),
    (1.8, 1.3, 1.0, 1.1),
    (2.2, 1.5, 0.9, 1.0),
    (2.0, 1.1, 1.1, 1.2),
]


best_ndcg = -1
best_model = None
best_features = None
best_params = None


# =========================================================
# TUNING LOOP
# =========================================================

for alpha, beta, gamma, delta in param_grid:

    print("\nTesting:", alpha, beta, gamma, delta)

    Q = {}

    for i in range(n_features):

        score = (
            alpha * feature_cf[i]
            + beta * feature_sep[i]
            + delta * feature_cov[i]
            - gamma * redundancy[i]
        )

        Q[(i, i)] = -score

    BQM = dimod.BinaryQuadraticModel.from_qubo(Q)

    k = 75

    BQM.update(dimod.generators.combinations(n_features, k))

    sampler = SimulatedAnnealingSampler()

    sampleset = qa.submit(
        sampler,
        SimulatedAnnealingSampler.sample,
        BQM,
        label="1B-ensemble",
        num_reads=1500
    )

    best = sampleset.first.sample

    selected_features = [
        i for i, v in best.items() if v == 1
    ]

    ICM_selected = ICM.tocsc()[:, selected_features].tocsr()

    model = ItemKNNCBFRecommender(URM_train, ICM_selected)

    model.fit(
        topK=100,
        shrink=5,
        similarity='cosine',
        normalize=True
    )

    result_df, _ = evaluator.evaluateRecommender(model)

    ndcg = result_df.loc[10, "NDCG"]

    print("NDCG@10:", ndcg)

    if ndcg > best_ndcg:
        best_ndcg = ndcg
        best_model = model
        best_features = selected_features
        best_params = (alpha, beta, gamma, delta)


# =========================================================
# FINAL RESULTS
# =========================================================

print("\n================ BEST RESULT ================")
print("Best NDCG@10:", best_ndcg)
print("Best params (alpha, beta, gamma, delta):", best_params)
print("Selected features:", len(best_features))

EvaluatorHoldout: Ignoring 3 ( 0.0%) Users that have less than 1 test interactions

Testing: 2.5 1.2 1.0 1.0
ItemKNNCBFRecommender: URM Detected 23 ( 0.2%) items with no interactions.
ItemKNNCBFRecommender: ICM Detected 98 ( 0.7%) items with no features.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 14607 (100.0%), 3271.55 column/sec. Elapsed time 4.46 sec
EvaluatorHoldout: Processed 20425 (100.0%) in 10.37 sec. Users per second: 1970
NDCG@10: 0.03368006686987563

Testing: 2.0 1.0 1.2 1.0
ItemKNNCBFRecommender: URM Detected 23 ( 0.2%) items with no interactions.
ItemKNNCBFRecommender: ICM Detected 98 ( 0.7%) items with no features.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 14607 (100.0%), 3277.56 column/sec. Elapsed time 4.46 sec
EvaluatorHoldout: Processed 20425 (100.0%) in 10.43 sec. Users per second: 1959
NDCG@10: 0.034167709284201354

Testing: 1.8 1.3 1.0 1.1
ItemKNNCBFRecommender: URM Detected 23 ( 0.2%) items